# 获取结构化结果的方式

## 1、使用with_structured_output

这种方式是 最新 、 最简洁 的API，直接让模型“理解”你需要的数据结构，并返回解析好的对象。

此外，我们可以在with_structured_output方法中传入 include_raw=True 参数，表示返回解析前的原始AIMessage ，从而访问令牌用量等元数据。


举例：

In [3]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os


# 1、提供大模型
load_dotenv(override=True)
DASHSCOPE_API_KEY = os.getenv("DASHSCOPE_API_KEY")
DASHSCOPE_BASE_URL = os.getenv("DASHSCOPE_BASE_URL")

model = init_chat_model(
    model="qwen3.7-plus",
    model_provider="openai",
    # temperature=1.5,
    api_key=DASHSCOPE_API_KEY,
    base_url=DASHSCOPE_BASE_URL,
    extra_body={"enable_thinking": False},
    # max_tokens=10,
)

In [6]:
from pydantic import BaseModel, Field
from rich import print as rprint

class Movie(BaseModel):
    """电影信息"""
    title: str = Field(description="电影标题")
    year: int = Field(description="上映年份")
    director: str = Field(description="导演")
    rating: float = Field(description="评分（10分制）")

# 设置模型结构化输出
model_with_structure = model.with_structured_output(Movie,include_raw=True)

# 调用模型并获取结构化输出
resp = model_with_structure.invoke("给我介绍下电影《星际穿越》")
print(type(resp))
rprint(resp)  # AIMessage

<class 'dict'>


{
    'raw': AIMessage(
        content='{\n  "title": "《星际穿越》 (Interstellar)",\n  "year": 2014,\n  "director": "克里斯托弗·诺兰 
(Christopher Nolan)",\n  "rating": 9.3\n}',
        additional_kwargs={
            'parsed': Movie(
                title='《星际穿越》 (Interstellar)',
                year=2014,
                director='克里斯托弗·诺兰 (Christopher Nolan)',
                rating=9.3
            ),
            'refusal': None
        },
        response_metadata={
            'token_usage': {
                'completion_tokens': 53,
                'prompt_tokens': 22,
                'total_tokens': 75,
                'completion_tokens_details': None,
                'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 22}
            },
            'model_provider': 'openai',
            'model_name': 'qwen3.7-plus',
            'system_fingerprint': None,
            'id': 'chatcmpl-f63f045c-4618-9eec-a25b-25f47ff6b19b',
            'finish_reason': 'stop',
            'logprobs': None
        },
        id='lc_run--019f5fc8-f4ee-7c20-9273-70282a7159f7-0',
        tool_calls=[],
        invalid_tool_calls=[],
        usage_metadata={
            'input_tokens': 22,
            'output_tokens': 53,
            'total_tokens': 75,
            'input_token_details': {'cache_read': 0},
            'output_token_details': {}
        }
    ),
    'parsed': Movie(
        title='《星际穿越》 (Interstellar)',
        year=2014,
        director='克里斯托弗·诺兰 (Christopher Nolan)',
        rating=9.3
    ),
    'parsing_error': None
}

输出包含了完整的输出响应，包含三个字段：

raw：返回的原始AIMessage。

parsed：解析后的输出

parsing_error：解析错误，当前用的是Pydantic，校验，格式不符合schema会导致报错。其它三种方式不符合schema不会导致报错。


In [10]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 1. 创建提示词模板
prompt_template = ChatPromptTemplate.from_messages([
    ("system","回答用户问题,必须始终输出一个包含title(电影标题)和year(上映年份)的 JSON 对象"),
    ("human","问题：{question}")
])

# 2. 模型初始化
# 从.env文件中加载环境变量
load_dotenv(override=True)
DASHSCOPE_API_KEY = os.getenv("DASHSCOPE_API_KEY")
DASHSCOPE_BASE_URL = os.getenv("DASHSCOPE_BASE_URL")

model = init_chat_model(
    model="qwen3.7-plus",
    model_provider="openai",
    # temperature=1.5,
    api_key=DASHSCOPE_API_KEY,
    base_url=DASHSCOPE_BASE_URL,
    extra_body={"enable_thinking": False},
    # max_tokens=10,
)

# 3. 定义结构
class Movie(BaseModel):
    """电影信息"""
    title: str = Field(description="电影标题")
    year: int = Field(description="上映年份")

# 4. 创建输出解析器
parser = JsonOutputParser(pydantic_object=Movie)

# 5. 创建链
chain = prompt_template | model | parser

# 6. 调用（返回字典）
# response = chain.invoke({"question": "介绍电影《盗梦空间》"})

response = parser.invoke(model.invoke(prompt_template.invoke({"question":"介绍电影《盗梦空间》"})))
print(type(response))
print(response)

<class 'dict'>
{'title': '盗梦空间', 'year': 2010}


## 2、使用输出解析器（了解）

这种方法更传统，依赖于在提示词中明确指示模型输出特定格式的文本，然后使用解析器进行转换。

其流程是： 提示词指导 (引导生成指定类型） → 模型生成文本 → 解析器转换
